In [ ]:
import mercury as mr
import pandas as pd
import re
import numpy as np
import plotnine
from scipy import misc
#import celery
from plotnine import *
import matplotlib as mpl
from IPython.display import display, HTML

In [ ]:
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2E}'.format)
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R 
suppressMessages(library(ggplot2))
suppressMessages(library(magrittr))
suppressMessages(library(dplyr))
source("ggtranscriptBase2.R")
source("/lustre/projects/Research_Project-MRC148213/sl693/scripts/ggtranscript/R/shorten_gaps.R")

In [ ]:
app = mr.App(title="Long-read Transcriptome Resource", description="ONT long-read dataset - Mill2023")

In [ ]:
mr.Markdown(text="""
# **Welcome to our ONT long-read transcriptome resource!**

Our study leverages both whole and targeted long read transcriptome sequencing data in 47 human cortex samples which span from prenatal to postnatal ages (6 wpc to 95 years). Please refer to Bamford et al. (2024) for more details.
Note: current resource only generates annotations and plots for known genes. 
""")

In [ ]:
dir = "/lustre/projects/Research_Project-MRC148213/sl693/website/"
classFileName = "classfile.csv"
numGenes = "numGenes.csv"
WholeGrouDTEName = "filtered_output_whole_group_removinglateprenatal_TEST.csv"
WholeGroupDTESexName = "filtered_output_whole_sex_removinglateprenatal_TEST.csv"
WholeGroupDTENormName = "filtered_whole_group_norm.csv"
WholeGroupDTESexNormName = "filtered_whole_sex_norm.csv"
WholeGroupDGEName = "whole_group_gene_norm.csv"
WholeGroupDGEResultsName = "DGEGroup.csv"
WholeSexDGEResultsName = "DGESex.csv"
#gtfName = "WholeTargeted_cleaned_aligned_merged_collapsed_qced_corrected_2reads2samples_2reads2samples_nomonointergenic.gtf"
gtfName = "knownGenesExons.gtf"
refgtfName = "refExons.gtf"
phenotypeName = "phenotype.csv"

In [ ]:
#classFile = pd.read_csv(classFileName)
allClassFileNum = pd.read_csv(numGenes)
WholeGroupDTE = pd.read_csv(WholeGrouDTEName)
WholeGroupDTESex = pd.read_csv(WholeGroupDTESexName)
phenotype = pd.read_csv(phenotypeName, dtype={'sample': 'string','group': 'string','sex': 'string','Weight': 'float64'})

In [ ]:
def subset_file(file, num, pattern):
    header = []
    matching_lines = []
    with open(file, 'r') as file:
        for ele, line in enumerate(file):
            if ele == 0:
                #line = line.replace('\n','')
                header.append(line)
            if pattern == line.split(",")[num]:
                line = line.replace('\n','')
                matching_lines.append(line)
    results = pd.DataFrame([l.split(",") for l in matching_lines])
    if len(results) > 0:
        results.columns = [h.split(",") for h in header][0]
    return(results)

In [ ]:
def subset_file_num(file, num, pattern):
    header = []
    matching_lines = []
    with open(file, 'r') as file:
        for ele, line in enumerate(file):
            if ele == 0:
                line = line.replace('\n','')
                header.append(line)
            else:                    
                if pattern == int(line.split(",")[num]):
                    line = line.replace('\n','')
                    matching_lines.append(line)
    results = pd.DataFrame([l.split(",") for l in matching_lines])
    if len(results) > 0:
        results.columns = [h.split(",") for h in header]
    return(results)

In [ ]:
name = mr.Text(label="Type ENSEMBL gene name:")

if name.value: 
    
    if name.value in list(allClassFileNum["associated_gene"]):

        mr.Md(f"""## {name.value}""")
        classFileNum = allClassFileNum[allClassFileNum["associated_gene"] == name.value]

        classFile = subset_file(classFileName, 1, name.value)
        #classFile.columns = ["isoform","associated_gene","structural_cateogory","associated_transcript","length","exons","key"]

        #geneFile = classFile[classFile["associated_gene"] == "APP"]
        #NovelTranscripts = geneFile[~geneFile["structural_category"].isin(["FSM", "ISM"])].shape[0]
        AllTranscripts = classFileNum["totalN"].values[0]
        NovelTranscripts = classFileNum["novelN"].values[0]
        DTETranscripts = list(set(list(classFile["isoform"])).intersection(list(WholeGroupDTE["isoform"])))
        DTESexTranscripts = list(set(list(classFile["isoform"])).intersection(list(WholeGroupDTESex["isoform"])))
        DTE = WholeGroupDTE[WholeGroupDTE["isoform"].isin(DTETranscripts)].sort_values(by = "padj")
        DTESex = WholeGroupDTESex[WholeGroupDTESex["isoform"].isin(DTESexTranscripts)].sort_values(by = "padj")

        # display Markdown
        mr.Md(f"Total number of transcripts: {AllTranscripts}")
        mr.Md(f"Number of novel transcripts: {NovelTranscripts}")
    
    else:
        mr.Md(f"#### {name.value} is not detected in our dataset.")
        mr.Md(f"Please check the spelling of input gene or insert another gene of interest.")
        

In [ ]:
def plot_expression(dat):
    # Create a DataFrame 'dat' and convert 'group' to a categorical variable
    dat['group'] = pd.Categorical(dat['group'], categories=['Prenatal', 'Postnatal'], ordered=True)
    dat["age"] = pd.to_numeric(dat["age"])
    dat["normalised_counts"] = pd.to_numeric(dat["normalised_counts"])
    
    # Define age scaling functions
    def fetal_ages_scale(age):
        return np.interp(age, (0, 40), (0, 40))

    def child_ages_scale(age):
        return np.interp(age, (0, 40), (41, 60))

    def adult_ages_scale(age):
        return np.interp(age, (40, 100), (75, 100))
    


    # Apply the scaling functions to create 'age.rescale' column
    dat['age.rescale'] = np.nan
    dat['age.rescale'] = np.where(dat['group'] == 'Prenatal', dat['age'].apply(fetal_ages_scale), dat['age.rescale'])
    dat['age.rescale'] = np.where((dat['group'] == 'Postnatal') & (dat['age'] <= 40), dat['age'].apply(child_ages_scale), dat['age.rescale'])
    dat['age.rescale'] = np.where((dat['group'] == 'Postnatal') & (dat['age'] > 40), dat['age'].apply(adult_ages_scale), dat['age.rescale'])
    
    breaks = [dat['age.rescale'].iloc[i] for i in [0, 14, 30, 31, 32, 39, 19]]
    labels = [dat['age'].iloc[i] for i in [0, 14, 30]] + ['40/0'] + [dat['age'].iloc[i] for i in [32, 39, 19]]

    width, height = 15, 6 
    x_vline = 40

    # Create the plot
    gg = (
        ggplot(aes(x='age.rescale', y='normalised_counts'), data=dat) +
        geom_point(aes(fill = 'sex', colour = 'sex')) +
        labs(x=None, y='log10 normalized counts') +
        geom_smooth(method = "loess") +
        scale_x_continuous(breaks=breaks, labels=labels) +
        theme_classic() + 
        theme(figure_size=(width, height)) +
        geom_vline(xintercept=16,linetype="dashed")
    )

    x1 = dat['age.rescale'].min() + 4
    x2 = dat['age.rescale'].min() + 50
    y = dat['normalised_counts'].max() + 0.3

    # Add text annotations for "Pre-natal" and "Post-natal"
    gg = gg + annotate("text", x =x1, y = y, label = "Pre-natal") + annotate("text", x =x2, y = y, label = "Post-natal")

    return(gg)

                                   

In [ ]:
_ = mr.Note(text="Differential gene expression (DGE)")
DGEButton = mr.Checkbox(value=False, label="Plot gene expression", url_key="flag")

if name.value and DGEButton.value:
    mr.Markdown(text="""### Gene expression""")
    if len(subset_file(WholeGroupDGEResultsName, 0, name.value)) > 0: 
        mr.Md(f"Significant changes in gene expression across development")
    else:
        mr.Md(f"No significant changes in gene expression across development")
    
    if len(subset_file(WholeSexDGEResultsName, 0, name.value)) > 0: 
        mr.Md(f"Significant changes in gene expression with sex")
    else:
        mr.Md(f"No significant changes in gene expression with sex")
    
    pattern = name.value
    #pattern = "APP"
    norm = subset_file(WholeGroupDGEName, 1, name.value)
    matching_lines = []
    with open(WholeGroupDGEName, 'r') as file:
        for line in file:
            if pattern == line.split(",")[1]:
                matching_lines.append(line)
    dfNorm = pd.DataFrame([l.split(",") for l in matching_lines])
    dfNorm.columns = ["sample","gene","normalised_counts","group","sex","age","time","time"]   
    genePlot = plot_expression(dfNorm)
    print(genePlot)
else:
    pass

In [ ]:
_ = mr.Note(text="Differential transcript expression (DTE)")
DTEButton = mr.Checkbox(value=False, label="Show table", url_key="flag")
    
if name.value and DTEButton.value:
    mr.Markdown(text="""### Differentially expressed transcripts""")
    mr.Md(f"Number of differentially expressed transcripts across development: {len(DTETranscripts)}")
    mr.Md(f"Number of differentially expressed transcripts by sex: {len(DTESexTranscripts)}")
    
    dat1 = classFile[classFile["isoform"].isin(DTETranscripts)].set_index("isoform")
    dat2 = WholeGroupDTE[WholeGroupDTE["isoform"].isin(DTETranscripts)].set_index("isoform")[["log2FoldChange","pvalue","padj"]]
    #x = dat2.join(dat1.drop(columns=['key']), how='left') 
    x = dat2.join(dat1, how='left') 
    if len(x) > 0:
        display(HTML("<div style='height: 200px'>" + x.style.render() + "</div>"))
    
else:
    pass

In [ ]:
def transcript_output(pattern, dat):
    
    # isoKeys = dict(zip(list(classFile["isoform"].values.ravel()), list(classFile["key"].values.ravel())))
    # isoKeys.get("ONT11_2781_9724")
    dat1 = classFile[classFile["isoform"] == pattern].set_index("isoform")
    dat2 = WholeGroupDTE[WholeGroupDTE["isoform"] == pattern].set_index("isoform")[["log2FoldChange","pvalue","padj"]]
    #x = dat2.join(dat1.drop(columns=['key']), how='left') 
    x = dat2.join(dat1, how='left') 
    xselect = x.loc[[pattern]]
    display(HTML("<div style='height: 200px'>" + xselect.style.render() + "</div>"))
        
    #key = int(dat1["key"].values[0])
    #norm = subset_file(WholeGroupDTENormName, 0, pattern).T.drop(index="key")
    norm = subset_file(WholeGroupDTENormName, 0, pattern).T.drop(index="isoform")
    norm.columns = ["normalised_counts"]
    norm = pd.merge(norm, phenotype, left_index=True, right_index=False, right_on = "sample")
    transcriptPlot = plot_expression(norm)
    
    return(transcriptPlot)
    
def extract_gtf(pattern, gtfInput, type):
    matching_lines = []
    with open(gtfInput, 'r') as file:
        for line in file:
            if type == "reference":            
                if pattern == line.split(",")[1]:
                    line = line.replace('\n','')
                    matching_lines.append(line)
            else:
                if pattern == line.split(",")[0]:
                    line = line.replace('\n','')
                    matching_lines.append(line)
    gtfExtract = pd.DataFrame([l.split(",") for l in matching_lines])
    if type == "reference":
         gtfExtract.columns = ["transcript_id","gene_id","seqnames","strand","start","end"] 
    else:
         gtfExtract.columns = ["transcript_id","gene_id","seqnames","strand","start","end"] 
    gtfExtract["start"] = [int(i) for i in gtfExtract["start"]]
    gtfExtract["end"] = [int(i) for i in gtfExtract["end"]]
    return(gtfExtract)

def get_type(transcript_id):
    if "ENST" in transcript_id:
        return 'Reference'
    else:
        return 'LR'
    
def plot_structure(transcript, gene):
    gtfExtract = extract_gtf(transcript, gtfName,"LR")
    rgtfExtract = extract_gtf(gene, refgtfName,type="reference")
    merged = pd.concat([gtfExtract,rgtfExtract])
    merged['Type'] = merged['transcript_id'].apply(get_type)
    return(merged)

In [ ]:
merged = pd.DataFrame()
DTEPlotButton = mr.Checkbox(value=False, label="Plot transcript expression", url_key="flag")
    
if name.value and DTEPlotButton.value:
    selectedGroup = mr.Select(label="across development: pre-natal vs post-natal", 
                              choices=list(DTE["isoform"]))

    if selectedGroup.value is not None:
        mr.Md(f"""###{selectedGroup.value}""")
        mr.Md(f"""#### Summary""")
        tab = transcript_output(selectedGroup.value,WholeGroupDTENormName)
        #merged = plot_structure("ONT21_1927_4635","APP")
        mr.Md(f"""#### Transcript structure""")
        merged = plot_structure(selectedGroup.value, name.value)
else:
    pass

In [ ]:
%%R -i merged 
#IRkernel::set_plot_options(width=600,units="px")
if (nrow(merged) > 0) {
    gexons = merged
    gintrons = gexons %>% to_intron(group_var = "transcript_id")
    grescaled = shorten_gaps(gexons, gintrons, group_var = "transcript_id")

    p <- grescaled %>%
      dplyr::filter(type == "exon") %>%
      ggplot(aes(xstart = start,xend = end,y = transcript_id)) +
      geom_range(aes(fill = Type)) +
      labs(y ="") + 
      geom_intron(data = grescaled %>% dplyr::filter(type == "intron"),arrow.min.intron.length = 300) +
      theme_classic() +
      theme(legend.position = "None", 
        axis.line.x = element_line(colour = "grey80"),
        panel.background = element_rect(fill = "white", colour = "grey50"),
        panel.border = element_rect(fill = NA, color = "grey50", linetype = "dotted"),
        axis.text.y= element_text(size=12),
        strip.text.y = element_text(size = 12, color = "black"),
        strip.background = element_rect(fill = "white", colour = "grey50")) + scale_fill_manual(values = c("red","black"))
    
    p
    
}

In [ ]:
if name.value and DTEPlotButton.value and selectedGroup.value is not None:
    mr.Md(f"""#### Transcript expression""")
    print(tab)

In [ ]:
mergedSex = pd.DataFrame()
if name.value and DTEPlotButton.value: 
    selectedSex = mr.Select(label="across sex: female vs male", 
                          choices=list(DTESex["isoform"]))

    if selectedSex.value is not None:
        transcript_output(selectedSex.value,WholeGroupDTESexNormName)
        mergedSex = plot_structure(selected.value, name.value)
else:
    pass

In [ ]:
%%R -i mergedSex

if (nrow(mergedSex) > 0) {
    gexons = mergedSex
    gintrons = gexons %>% to_intron(group_var = "transcript_id")
    grescaled = shorten_gaps(gexons, gintrons, group_var = "transcript_id")

    grescaled %>%
      dplyr::filter(type == "exon") %>%
      ggplot(aes(xstart = start,xend = end,y = transcript_id)) +
      geom_range(aes(fill = Type)) +
      labs(y ="") + 
      geom_intron(data = grescaled %>% dplyr::filter(type == "intron"),arrow.min.intron.length = 300) +
      theme_classic() +
      theme(legend.position = "None", 
        axis.line.x = element_line(colour = "grey80"),
        panel.background = element_rect(fill = "white", colour = "grey50"),
        panel.border = element_rect(fill = NA, color = "grey50", linetype = "dotted"),
        axis.text.y= element_text(size=12),
        strip.text.y = element_text(size = 12, color = "black"),
        strip.background = element_rect(fill = "white", colour = "grey50")) + scale_fill_manual(values = c("red","black")) 
}

In [ ]:
#classFileNum = allClassFileNum[allClassFileNum["associated_gene"] == "AMBRA1"]
#classFile = subset_file(classFileName, 1, "AMBRA1")
#AllTranscripts = classFileNum["totalN"].values[0]
#NovelTranscripts = classFileNum["novelN"].values[0]
#DTETranscripts = list(set(list(classFile["isoform"])).intersection(list(WholeGroupDTE["isoform"])))
#DTESexTranscripts = list(set(list(classFile["isoform"])).intersection(list(WholeGroupDTESex["isoform"])))
#DTE = WholeGroupDTE[WholeGroupDTE["isoform"].isin(DTETranscripts)].sort_values(by = "padj")
#DTESex = WholeGroupDTESex[WholeGroupDTESex["isoform"].isin(DTESexTranscripts)].sort_values(by = "padj")
#dat1 = classFile[classFile["isoform"].isin(DTETranscripts)].set_index("isoform")
#dat2 = WholeGroupDTE[WholeGroupDTE["isoform"].isin(DTETranscripts)].set_index("isoform")[["log2FoldChange","pvalue","padj"]]
#x = dat2.join(dat1, how='left') 
#x

In [ ]:
#%load_ext line_profiler
#%lprun -f transcript_output transcript_output("ONT11_2781_9724",WholeGroupDTENormName)